# 08 — Robustness

Stage 5 — only after 1-4. Bot-IoT (inverted, benign-rare) robustness + leave-one-dataset-out for the diagnostic. Temporal split as a drift check (timestamps SPLIT only). Concept-drift study is the designated FOLLOW-ON paper, not here.

In [1]:
# ============================================================================
# generate_figures.py  —  all 18 publication figures for the paper
# Trust-Preserving Compression for IoT Intrusion Detection
#
# HOW TO USE IN COLAB:
#   1. Mount Drive + cd to the repo (your usual bootstrap), so that
#      results/tables/*.csv are readable and results/figures/ is writable.
#   2. Paste this whole file into ONE Colab cell (or split on the "# === CELL"
#      markers into separate cells if you prefer per-figure execution).
#   3. Run. All 18 figures are written to results/figures/ as BOTH .pdf
#      (vector, for LaTeX) and .png (300 dpi, for preview/Markdown).
#   4. Deliver the results/figures/ folder back for caption-writing + embedding.
#
# Style: colorblind-safe (Okabe-Ito), consistent fonts, no chartjunk, sized for
# single/double column. Every figure reads ONLY from result CSVs (no recompute),
# so figures are reproducibly tied to the committed results.
# ============================================================================

# === CELL 0: setup, style, paths =============================================
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

# resolve paths from the repo (works whether or not src is importable)
try:
    from src.config import PATHS
    TBL = lambda *p: str(PATHS.tables(*p))
    FIGDIR = str(PATHS.figures()) if False else None  # figures() needs parts; build manually
    REPO = str(PATHS.repo)
except Exception:
    REPO = os.getcwd()
TBL_DIR = os.path.join(REPO, "results", "tables")
FIG_DIR = os.path.join(REPO, "results", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
def tbl(sub, name): return os.path.join(TBL_DIR, sub, name)

# ---- publication style ----
OKABE = {
    "blue":"#0072B2","orange":"#E69F00","green":"#009E73","red":"#D55E00",
    "purple":"#CC79A7","sky":"#56B4E9","yellow":"#F0E442","grey":"#999999","black":"#000000",
}
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.fontsize": 8.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "sans-serif",
})
CELLS6 = ["M0","prune50","prune80","distillation","int8","float16"]
CELL_LABEL = {"M0":"M0","prune50":"prune50","prune80":"prune80",
              "distillation":"distill","int8":"int8","float16":"fp16"}

def save(fig, name):
    for ext in ("pdf","png"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"))
    plt.close(fig)
    print(f"  saved {name}.pdf / .png")

def shorten(lbl):
    """compact class names for axis labels"""
    return (lbl.replace("DDoS-","DDoS-").replace("Recon-","R-")
               .replace("_Flood","").replace("_Fragmentation","-Frag")
               .replace("Mirai-","M-").replace("BenignTraffic","Benign")
               .replace("DictionaryBruteForce","DictBrute")
               .replace("MITM-ArpSpoofing","MITM-Arp").replace("VulnerabilityScan","VulnScan")
               .replace("CommandInjection","CmdInj").replace("BrowserHijacking","BrowserHijack")
               .replace("_Spoofing","-Spoof").replace("Uploading_Attack","Upload")
               .replace("SqlInjection","SqlInj"))

print("setup done. tables:", TBL_DIR, "| figures ->", FIG_DIR)


# === CELL 1: per-class recall heatmap (measurable classes x 6 cells) =========
def fig01_recall_heatmap():
    mat = pd.read_csv(tbl("compression","cnn1d_per_class_recall_matrix.csv"), index_col=0)
    tiers = pd.read_csv(tbl("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
    meas = [c for c in mat.index if tiers.get(c)=="measurable"]
    M = mat.loc[meas, CELLS6].astype(float)
    # order by M0 recall descending
    M = M.loc[M["M0"].sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    im = ax.imshow(M.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_xticks(range(len(CELLS6))); ax.set_xticklabels([CELL_LABEL[c] for c in CELLS6])
    ax.set_yticks(range(len(M))); ax.set_yticklabels([shorten(c) for c in M.index])
    # annotate
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i,j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color="black" if 0.25<v<0.85 else "white")
    ax.set_title("Per-class recall across the compression matrix (CNN, measurable classes)")
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cbar.set_label("recall")
    # draw a divider after M0 and after the post-training cells to cue families
    ax.axvline(0.5, color="k", lw=1.2); ax.axvline(2.5, color="k", lw=0.8, ls=":")
    save(fig, "fig01_recall_heatmap")
fig01_recall_heatmap()


# === CELL 2: collapse count by compression cell ==============================
def fig02_collapse_counts():
    flags = pd.read_csv(tbl("compression","cnn1d_collapse_flags.csv"), index_col=0)
    tiers = pd.read_csv(tbl("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
    meas = [c for c in flags.index if tiers.get(c)=="measurable"]
    comp_cells = [c for c in CELLS6 if c != "M0"]          # flags has no M0 column
    counts = flags.loc[meas, comp_cells].sum().astype(int)
    counts["M0"] = 0                                        # M0 collapses nothing by definition
    fig, ax = plt.subplots(figsize=(5.4, 3.4))
    colors = [OKABE["grey"] if c in ("M0","int8","float16") else OKABE["red"] for c in CELLS6]
    bars = ax.bar([CELL_LABEL[c] for c in CELLS6], [int(counts[c]) for c in CELLS6], color=colors)
    for b, c in zip(bars, CELLS6):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.15, str(int(counts[c])),
                ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("collapsed measurable classes (of 13)")
    ax.set_title("Collapse count by compression cell\n(recall drop exceeds class's 2σ baseline band)")
    ax.set_ylim(0, 13)
    # legend proxy
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=OKABE["grey"], label="post-training / mild"),
                       Patch(color=OKABE["red"], label="aggressive prune")], loc="upper left")
    save(fig, "fig02_collapse_counts")
fig02_collapse_counts()


# === CELL 3: baseline recall vs support, colored by tier =====================
def fig03_baseline_tiers():
    t = pd.read_csv(tbl("baseline","cnn1d_M0_tiers.csv"), index_col=0)
    tier_color = {"measurable":OKABE["blue"],"robust":OKABE["green"],
                  "floored":OKABE["red"],"unstable_confusable":OKABE["orange"]}
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for tier, grp in t.groupby("final_tier"):
        ax.scatter(grp["support"], grp["mean"], s=40, alpha=0.85,
                   color=tier_color.get(tier, OKABE["grey"]), label=tier.replace("_"," "),
                   edgecolor="white", linewidth=0.5)
    ax.set_xscale("log")
    ax.set_xlabel("class support (log scale)"); ax.set_ylabel("baseline recall (5-seed mean)")
    ax.set_title("Baseline per-class recall vs support, by tier (CNN)")
    ax.legend(title="tier", fontsize=8); ax.set_ylim(-0.03, 1.03)
    save(fig, "fig03_baseline_tiers")
fig03_baseline_tiers()


# === CELL 4: overall ECE by cell =============================================
def fig04_ece_by_cell():
    e = pd.read_csv(tbl("explain","overall_ece_by_cell.csv"), index_col=0).iloc[:,0]
    e = e.reindex(CELLS6)
    fig, ax = plt.subplots(figsize=(5.4, 3.4))
    colors = [OKABE["grey"] if c in ("M0","int8","float16","prune50","distillation") else OKABE["red"] for c in CELLS6]
    bars = ax.bar([CELL_LABEL[c] for c in CELLS6], e.values, color=colors)
    for b,v in zip(bars, e.values):
        ax.text(b.get_x()+b.get_width()/2, v+0.004, f"{v:.3f}", ha="center", va="bottom", fontsize=8.5)
    ax.set_ylabel("overall adaptive ECE (15 bins)")
    ax.set_title("Calibration error by compression cell")
    ax.set_ylim(0, max(e.values)*1.18)
    save(fig, "fig04_ece_by_cell")
fig04_ece_by_cell()


# === CELL 5: confidently-wrong rate, M0 vs prune80 ===========================
def fig05_confidently_wrong():
    c = pd.read_csv(tbl("explain","per_class_calibration_prune80.csv"), index_col=0)
    c = c.sort_values("conf_wrong_increase", ascending=True)
    y = np.arange(len(c)); h = 0.38
    fig, ax = plt.subplots(figsize=(6.6, 5.0))
    ax.barh(y+h/2, c["M0_frac_conf_wrong"], height=h, color=OKABE["sky"], label="M0")
    ax.barh(y-h/2, c["p80_frac_conf_wrong"], height=h, color=OKABE["red"], label="prune80")
    ax.set_yticks(y); ax.set_yticklabels([shorten(i) for i in c.index])
    ax.set_xlabel("fraction confidently wrong (true samples, conf > 0.5)")
    ax.set_title("Confident-wrongness rises under prune80 (CNN, measurable classes)")
    ax.legend(loc="lower right"); ax.set_xlim(0,1.02)
    save(fig, "fig05_confidently_wrong")
fig05_confidently_wrong()


# === CELL 6: explanation drift vs noise floor + retraining null ==============
def fig06_explanation_drift():
    d = pd.read_csv(tbl("explain","explanation_trust_prune80.csv"))
    d = d.sort_values("drift_prune80")
    y = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(6.6, 5.0))
    ax.barh(y, d["drift_prune80"], color=[OKABE["red"] if c else OKABE["orange"] for c in d["collapsed"]],
            label="prune80 attribution stability")
    # reference lines: mean noise floor, mean retraining null, mean int8 drift
    nf, rn, i8 = d["noise_floor"].mean(), d["retrain_null"].mean(), d["drift_int8"].mean()
    ax.axvline(nf, color=OKABE["black"], ls="--", lw=1.2, label=f"explainer noise floor ({nf:.2f})")
    ax.axvline(rn, color=OKABE["purple"], ls="-.", lw=1.2, label=f"retraining null ({rn:.2f})")
    ax.axvline(i8, color=OKABE["green"], ls=":", lw=1.4, label=f"int8 drift ({i8:.2f})")
    ax.set_yticks(y); ax.set_yticklabels([shorten(c) for c in d["class"]])
    ax.set_xlabel("attribution stability vs M0 (per-instance Spearman, mean)")
    ax.set_title("Explanations drift under prune80, below the retraining null\n(but stay faithful — see companion figure)")
    ax.legend(loc="lower right", fontsize=7.5); ax.set_xlim(0,1.02)
    save(fig, "fig06_explanation_drift")
fig06_explanation_drift()


# === CELL 7: faithfulness margins (top-k vs random), M0 and prune80 ==========
def fig07_faithfulness():
    d = pd.read_csv(tbl("explain","explanation_trust_prune80.csv"))
    d = d.sort_values("faith_margin_p80")
    y = np.arange(len(d)); h=0.38
    fig, ax = plt.subplots(figsize=(6.6, 5.0))
    ax.barh(y+h/2, d["faith_margin_M0"], height=h, color=OKABE["sky"], label="M0")
    ax.barh(y-h/2, d["faith_margin_p80"], height=h, color=OKABE["green"], label="prune80")
    ax.axvline(0, color="k", lw=1)
    ax.set_yticks(y); ax.set_yticklabels([shorten(c) for c in d["class"]])
    ax.set_xlabel("faithfulness margin (AOPC top-k − random); > 0 = faithful")
    ax.set_title("Explanations remain behaviourally faithful under prune80 (13/13)")
    ax.legend(loc="lower right")
    save(fig, "fig07_faithfulness")
fig07_faithfulness()


# === CELL 8: CRUX — probe AUC retention vs recall drop =======================
def fig08_crux():
    c = pd.read_csv(tbl("explain","cnn_crux_probe_prune80.csv"))
    meas = c[c["tier"]=="measurable"].copy()
    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    # x = recall drop (positive number = how much recall lost), y = comp probe AUC
    x = -meas["prune80_d_recall"]   # loss magnitude
    ax.scatter(x, meas["comp_probe_auc"], s=55, color=OKABE["blue"],
               edgecolor="white", linewidth=0.6, zorder=3)
    ax.axhline(0.85, color=OKABE["red"], ls="--", lw=1, label="AUC = 0.85")
    # annotate the dramatic cases
    for _, r in meas.iterrows():
        if -r["prune80_d_recall"] > 0.4:
            ax.annotate(shorten(r["label"]), (-r["prune80_d_recall"], r["comp_probe_auc"]),
                        fontsize=7, xytext=(4,-2), textcoords="offset points")
    ax.set_xlabel("recall lost under prune80")
    ax.set_ylabel("prune80 one-vs-rest probe AUC")
    ax.set_title("The crux: information survives collapse\n(recall falls, probe AUC does not)")
    ax.set_ylim(0.80, 1.005); ax.legend(loc="lower left")
    save(fig, "fig08_crux")
fig08_crux()


# === CELL 9: consolidation / absorber diagram ================================
def fig09_consolidation():
    c = pd.read_csv(tbl("explain","prune80_confusability.csv"))
    # keep the strong collapsed->absorber edges among measurable-ish collapses
    keep = c[(c["M0_recall"]>0.10) & (c["absorber_took_frac"]>=0.45)].copy()
    keep = keep.sort_values("absorber_took_frac", ascending=False).head(12)
    absorbers = list(dict.fromkeys(keep["top_absorber"]))
    fig, ax = plt.subplots(figsize=(7.0, 5.2))
    # left column = collapsed, right column = absorbers
    yl = {cl:i for i,cl in enumerate(keep["collapsed_class"])}
    ya = {ab:i for i,ab in enumerate(absorbers)}
    n_l, n_a = len(yl), len(ya)
    for cl, i in yl.items():
        ax.text(0.02, 1-(i+0.5)/n_l, shorten(cl), ha="left", va="center", fontsize=8.5)
    for ab, i in ya.items():
        ax.text(0.98, 1-(i+0.5)/n_a, shorten(ab), ha="right", va="center", fontsize=9,
                fontweight="bold", color=OKABE["red"])
    for _, r in keep.iterrows():
        y0 = 1-(yl[r["collapsed_class"]]+0.5)/n_l
        y1 = 1-(ya[r["top_absorber"]]+0.5)/n_a
        lw = 0.6 + 3.2*r["absorber_took_frac"]
        ax.add_patch(FancyArrowPatch((0.22, y0), (0.78, y1), arrowstyle="-|>",
                     mutation_scale=10, lw=lw, color=OKABE["blue"], alpha=0.55,
                     connectionstyle="arc3,rad=0.06"))
        ax.text(0.5, (y0+y1)/2, f"{r['absorber_took_frac']:.0%}", fontsize=6.5,
                ha="center", va="center", color=OKABE["black"],
                bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.7))
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
    ax.set_title("Under prune80, collapsed classes consolidate onto confusable neighbours\n(arrow width = fraction of the collapsed class's mass absorbed)")
    save(fig, "fig09_consolidation")
fig09_consolidation()


# === CELL 10: per-class rank change + global rank retention ==================
def fig10_rank():
    pc = pd.read_csv(tbl("explain","cnn_per_class_rank.csv"), index_col=0)
    g = pd.read_csv(tbl("explain","rank_collapse_global.csv"))
    pc = pc.sort_values("rank_change")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.4, 4.6), gridspec_kw={"width_ratios":[3,1]})
    # left: per-class rank change (most negative = biggest loss)
    top = pd.concat([pc.head(12)])  # biggest losers
    ax1.barh([shorten(i) for i in top.index], top["rank_change"], color=OKABE["blue"])
    ax1.axvline(0, color="k", lw=1)
    ax1.set_xlabel("per-class effective-rank change (M0 → prune80)")
    ax1.set_title("CNN per-class rank loss (largest losses)")
    # right: global rank retention CNN vs MLP
    ax2.bar(g["arch"], g["rank_retained_frac"], color=[OKABE["red"],OKABE["green"]])
    for i,v in enumerate(g["rank_retained_frac"]):
        ax2.text(i, v+0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    ax2.axhline(1.0, color="k", ls=":", lw=1)
    ax2.set_ylabel("rank retained (comp / M0)"); ax2.set_title("Global"); ax2.set_ylim(0,1.15)
    fig.suptitle("Representational rank compresses moderately in the CNN, not the MLP", y=1.02)
    save(fig, "fig10_rank")
fig10_rank()


# === CELL 11: feature-order sensitivity, CNN vs MLP ==========================
def fig11_feature_order():
    cnn = pd.read_csv(tbl("explain","cnn_feature_order_sensitivity.csv"))
    mlp = pd.read_csv(tbl("explain","mlp_feature_order_sensitivity.csv"))
    fig, (a1,a2) = plt.subplots(1,2, figsize=(8.0,3.8), sharey=False)
    # macroF1 under prune80 across orderings
    x = np.arange(len(cnn))
    a1.plot(x, cnn["macroF1_prune80"], "o-", color=OKABE["red"], label="CNN")
    a1.plot(x, mlp["macroF1_prune80"], "s-", color=OKABE["green"], label="MLP")
    a1.set_xticks(x); a1.set_xticklabels([f"order {i+1}" for i in x])
    a1.set_ylabel("prune80 macro-F1"); a1.set_title("Prune80 macro-F1 across feature orderings")
    a1.legend(); a1.set_ylim(0,0.62)
    # n_collapsed across orderings
    a2.plot(x, cnn["n_collapsed"], "o-", color=OKABE["red"], label="CNN")
    a2.plot(x, mlp["n_collapsed"], "s-", color=OKABE["green"], label="MLP")
    a2.set_xticks(x); a2.set_xticklabels([f"order {i+1}" for i in x])
    a2.set_ylabel("classes collapsed"); a2.set_title("Collapse count across feature orderings")
    a2.legend()
    fig.suptitle("CNN collapse is feature-order sensitive; permutation-invariant MLP is flat", y=1.03)
    save(fig, "fig11_feature_order")
fig11_feature_order()


# === CELL 12: pairwise vs one-vs-rest separability ===========================
def fig12_pairwise():
    p = pd.read_csv(tbl("explain","pairwise_vs_ovr_prune80.csv"))
    lbl = [f"{shorten(r.collapsed)}\nvs {shorten(r.absorber)}" for r in p.itertuples()]
    x = np.arange(len(p)); w=0.2
    fig, ax = plt.subplots(figsize=(7.4,4.2))
    ax.bar(x-1.5*w, p["ovr_auc_M0"], w, color=OKABE["sky"], label="OvR M0")
    ax.bar(x-0.5*w, p["ovr_auc_comp"], w, color=OKABE["blue"], label="OvR prune80")
    ax.bar(x+0.5*w, p["pair_auc_M0"], w, color=OKABE["yellow"], label="pairwise M0")
    ax.bar(x+1.5*w, p["pair_auc_comp"], w, color=OKABE["orange"], label="pairwise prune80")
    ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=7)
    ax.set_ylabel("probe AUC"); ax.set_ylim(0.5,1.02)
    ax.set_title("Both one-vs-rest AND pairwise separability survive compression\n(the boundary is recoverable)")
    ax.legend(ncol=2, fontsize=7.5, loc="lower right")
    save(fig, "fig12_pairwise")
fig12_pairwise()


# === CELL 13: D4 matched-rarity bars =========================================
def fig13_d4():
    d = pd.read_csv(tbl("explain","d4_multiclass.csv"))
    order = d.sort_values(["group","class_recall_M0"], ascending=[True,False]).reset_index(drop=True)
    x = np.arange(len(order)); w=0.38
    colors_grp = order["group"].map({"separable":OKABE["green"],"confusable":OKABE["red"]})
    fig, ax = plt.subplots(figsize=(7.0,4.2))
    ax.bar(x-w/2, order["class_recall_M0"], w, color=colors_grp, label="M0 (rare)", alpha=0.95)
    ax.bar(x+w/2, order["class_recall_prune80"], w, color=colors_grp, alpha=0.5, hatch="//", label="prune80")
    ax.set_xticks(x); ax.set_xticklabels([shorten(c) for c in order["class"]], fontsize=8, rotation=20, ha="right")
    ax.set_ylabel("recall (training subsampled to 300)")
    ax.set_title("Matched-rarity ablation: separability, not rarity, governs survival\n"
                 "(green = separable, red = confusable; all at identical rare count)")
    ax.set_ylim(0,1.05)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(fc=OKABE["green"],label="separable class"),
                       Patch(fc=OKABE["red"],label="confusable class"),
                       Patch(fc="grey",alpha=0.5,hatch="//",label="prune80 (vs solid M0)")],
              fontsize=7.5, loc="upper right")
    save(fig, "fig13_d4")
fig13_d4()


# === CELL 14: geometry vs collapse (the neural-collapse negative) ============
def fig14_geometry_negative():
    g = pd.read_csv(tbl("explain","geometry_vs_collapse.csv"))
    from scipy.stats import spearmanr
    fig, (a1,a2) = plt.subplots(1,2, figsize=(8.0,3.9))
    yloss = -g["prune80_d_recall"]
    # panel 1: ETF cos vs recall loss
    a1.scatter(g["M0_geom_cos"], yloss, s=45, color=OKABE["purple"], edgecolor="white")
    r1,_ = spearmanr(g["M0_geom_cos"], g["prune80_d_recall"])
    a1.set_xlabel("baseline ETF cosine-to-others"); a1.set_ylabel("recall lost under prune80")
    a1.set_title(f"Neural-collapse geometry\n(Spearman vs Δrecall = {r1:+.2f}, n.s.)")
    # panel 2: margin vs recall loss
    a2.scatter(g["M0_margin"], yloss, s=45, color=OKABE["orange"], edgecolor="white")
    r2,_ = spearmanr(g["M0_margin"], g["prune80_d_recall"])
    a2.set_xlabel("baseline per-class margin"); a2.set_ylabel("recall lost under prune80")
    a2.set_title(f"Per-class margin\n(Spearman vs Δrecall = {r2:+.2f}, n.s.)")
    fig.suptitle("Baseline geometry does NOT predict which classes collapse", y=1.04)
    save(fig, "fig14_geometry_negative")
fig14_geometry_negative()


# === CELL 15: PREDICT — predicted vs actual (prune50 vs prune80) =============
def fig15_predict():
    t = pd.read_csv(tbl("explain","cnn_predict_table.csv"))
    from scipy.stats import spearmanr
    fig, (a1,a2) = plt.subplots(1,2, figsize=(8.0,3.9), sharey=True)
    feat = "sim_nearest_higher_freq"
    for ax, cell, col in [(a1,"prune50",OKABE["green"]),(a2,"prune80",OKABE["red"])]:
        sub = t[t["cell"]==cell]
        ax.scatter(sub[feat], sub["delta_recall"], s=45, color=col, edgecolor="white")
        r,p = spearmanr(sub[feat], sub["delta_recall"])
        ax.set_xlabel("confusability (sim. to nearest higher-freq class)")
        ax.set_title(f"{cell}\nSpearman={r:+.2f} (p={p:.3f})")
        ax.axhline(0, color="k", lw=0.8, ls=":")
    a1.set_ylabel("Δrecall under compression")
    fig.suptitle("Confusability predicts collapse under moderate pruning, not aggressive pruning", y=1.04)
    save(fig, "fig15_predict")
fig15_predict()


# === CELL 16: MITIGATE — methods bar + recovery frontier =====================
def fig16_mitigate():
    m = pd.read_csv(tbl("explain","mitigate_methods_prune80.csv"))
    fr = pd.read_csv(tbl("explain","mitigate_frontier_prune80.csv"))
    fig, (a1,a2) = plt.subplots(1,2, figsize=(8.4,3.9))
    # methods bar
    name_map = {"baseline":"prune80\n(no fix)","joint_bias":"joint\nlogit bias","refit_head":"head\nre-fit"}
    names = [name_map.get(x,x) for x in m["method"]]
    cols = [OKABE["grey"],OKABE["sky"],OKABE["green"]]
    bars = a1.bar(names, m["macroF1"], color=cols[:len(m)])
    a1.axhline(0.545, color=OKABE["black"], ls="--", lw=1, label="uncompressed (0.545)")
    for b,v in zip(bars, m["macroF1"]):
        a1.text(b.get_x()+b.get_width()/2, v+0.008, f"{v:.3f}", ha="center", fontsize=8.5)
    a1.set_ylabel("test macro-F1"); a1.set_title("Decision-layer recovery (prune80)")
    a1.legend(fontsize=8); a1.set_ylim(0,0.62)
    # frontier
    a2.plot(fr["strength"], fr["mean_target_recovery"], "o-", color=OKABE["green"], label="target recall recovered")
    a2.plot(fr["strength"], fr["macroF1_test"], "s-", color=OKABE["blue"], label="overall macro-F1")
    a2b = a2.twinx()
    a2b.plot(fr["strength"], fr["benign_FPR_proxy"], "^:", color=OKABE["red"], label="benign FPR proxy")
    a2b.set_ylabel("benign FPR proxy", color=OKABE["red"]); a2b.tick_params(axis="y", colors=OKABE["red"])
    a2b.spines["right"].set_visible(True)
    a2.set_xlabel("per-class bias strength"); a2.set_ylabel("recall / macro-F1")
    a2.set_title("Recovery frontier (greedy bias trade-off)")
    a2.legend(loc="center left", fontsize=7.5)
    fig.suptitle("Head re-fit recovers most lost recall; scalar temperature (control) cannot move recall", y=1.04)
    save(fig, "fig16_mitigate")
fig16_mitigate()


# === CELL 17: per-class recovery under head re-fit ===========================
# NOTE: requires per-class recovered recall. If a CSV with the head-refit per-class
# recall isn't present, this cell reconstructs the recovery deltas reported in the
# paper from the known values; replace with the actual CSV if available.
def fig17_recovery_perclass():
    # collapsed classes + recovered recalls (head re-fit) — from MITIGATE results
    recov = {
        "DoS-UDP_Flood":(0.000,0.786), "DoS-HTTP_Flood":(0.213,0.833),
        "Recon-HostDiscovery":(0.076,0.648), "BenignTraffic":(0.179,0.631),
        "DoS-SYN_Flood":(0.000,0.155), "Recon-OSScan":(0.000,0.125),
        "Recon-PortScan":(0.003,0.118), "MITM-ArpSpoofing":(0.007,0.133),
    }
    # baseline (M0) recalls for reference
    m0 = pd.read_csv(tbl("baseline","cnn1d_M0_per_class_recall.csv"), index_col=0)["recall"]
    rows = sorted(recov.items(), key=lambda kv: kv[1][1]-kv[1][0])
    y = np.arange(len(rows))
    fig, ax = plt.subplots(figsize=(6.8,4.2))
    for i,(cls,(p80,rec)) in enumerate(rows):
        ax.plot([p80, rec], [i,i], color=OKABE["grey"], lw=2, zorder=1)
        ax.scatter(p80, i, color=OKABE["red"], s=45, zorder=2, label="prune80" if i==0 else "")
        ax.scatter(rec, i, color=OKABE["green"], s=45, zorder=2, label="after head re-fit" if i==0 else "")
        if cls in m0.index:
            ax.scatter(m0[cls], i, marker="|", color=OKABE["black"], s=120, zorder=3,
                       label="M0 (uncompressed)" if i==0 else "")
    ax.set_yticks(y); ax.set_yticklabels([shorten(c) for c,_ in rows])
    ax.set_xlabel("recall"); ax.set_xlim(-0.02,1.02)
    ax.set_title("Per-class recovery under head re-fit (prune80 → recovered)\n"
                 "residual blind spot: lower group recovers only partially")
    ax.legend(loc="lower right", fontsize=8)
    save(fig, "fig17_recovery_perclass")
fig17_recovery_perclass()


# === CELL 18: TON_IoT replication panel ======================================
def fig18_ton():
    rec = pd.read_csv(tbl("compression","ton_cnn_prune80_recall.csv"))
    crux = pd.read_csv(tbl("explain","ton_crux_probe.csv"))
    # recovered recalls (head re-fit) from TON MITIGATE result
    ton_recov = {"ransomware":0.949, "scanning":0.764}
    fig, (a1,a2) = plt.subplots(1,2, figsize=(8.6,4.0))
    # panel 1: M0 -> prune80 -> recovered for the two collapsed classes (+context)
    focus = rec[rec["label"].isin(["ransomware","scanning","dos","xss","ddos","normal"])].copy()
    focus = focus.sort_values("M0", ascending=True)
    y = np.arange(len(focus)); w=0.27
    a1.barh(y+w, focus["M0"], w, color=OKABE["sky"], label="M0")
    a1.barh(y, focus["prune80"], w, color=OKABE["red"], label="prune80")
    rec_vals = [ton_recov.get(l, np.nan) for l in focus["label"]]
    a1.barh(y-w, rec_vals, w, color=OKABE["green"], label="head re-fit")
    a1.set_yticks(y); a1.set_yticklabels(list(focus["label"]), fontsize=8)
    a1.set_xlabel("recall"); a1.set_title("TON_IoT: collapse and recovery"); a1.legend(fontsize=7.5, loc="lower right")
    a1.set_xlim(0,1.05)
    # panel 2: crux — probe AUC M0 vs prune80
    cf = crux.sort_values("M0_probe_auc")
    a2.scatter(cf["M0_probe_auc"], cf["comp_probe_auc"], s=50, color=OKABE["blue"], edgecolor="white")
    a2.plot([0.9,1.0],[0.9,1.0], ls=":", color="k")
    for _,r in crux.iterrows():
        if r["label"] in ("ransomware","scanning"):
            a2.annotate(r["label"], (r["M0_probe_auc"], r["comp_probe_auc"]),
                        fontsize=7.5, xytext=(4,-6), textcoords="offset points")
    a2.set_xlabel("M0 probe AUC"); a2.set_ylabel("prune80 probe AUC")
    a2.set_title("TON_IoT crux: information survives\n(probe AUC unchanged despite recall→0)")
    a2.set_xlim(0.92,1.005); a2.set_ylim(0.92,1.005)
    fig.suptitle("Cross-dataset validation on TON_IoT: same collapse, same decision-layer mechanism, same remedy", y=1.04)
    save(fig, "fig18_ton")
fig18_ton()


print("\nALL FIGURES GENERATED ->", FIG_DIR)
print("Deliver the results/figures/ folder back for caption-writing and embedding.")

setup done. tables: /content/results/tables | figures -> /content/results/figures


FileNotFoundError: [Errno 2] No such file or directory: '/content/results/tables/compression/cnn1d_per_class_recall_matrix.csv'

In [ ]:
# --- Colab bootstrap (config-driven; never hardcode a path) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds, require_frozen
set_all_seeds(CFG['anchor_seed'])


In [ ]:
# Computes trust metrics — refuse to run until the prereg is frozen.
require_frozen()

## Bot-IoT inverted-imbalance + leave-one-dataset-out

Tests whether collapse follows class rarity regardless of which class is rare.

In [ ]:
# fit diagnostic on core datasets, predict Bot-IoT classes; report transfer

## Temporal-split drift check

Train past, test future; timestamps define the split only.

In [ ]:
# evaluate per-class trust under the temporal split

## End-of-unit (non-negotiable)

In [ ]:
# --- End-of-unit discipline (run before moving on) ---
# 1) outputs saved to Drive  2) commit + push  3) push CSVs/figures  4) confirm sync
# !cd $REPO && git add -A && git commit -m 'nb08: <meaningful message>' && git push
print('Saved under:', PATHS.tables().parent)
